# 01 Region Setup

This notebook prepares the Marshfield coastal flood region. It reads the configured SFINCS bbox from the Location override, builds/reads the study AOI, collects required coastal static inputs, fetches coastal land/wave geometry, writes SSURGO infiltration rasters, and plots the collected products.

## Coastal Adaptation: Configured Bbox and Static Inputs

Marshfield does not draw a notebook bbox. The chosen SFINCS bbox is the configured `static_sources.bbox.output` GeoJSON. This notebook fails clearly if that file is missing or no longer contains the study area.

**Methodological Stages:**

### 1. Parameters and Configuration

Load the Marshfield Region config and show the configured bbox/static record syntax.

### 2. Configured Coastal Region

Build/read the study AOI, read the configured SFINCS bbox, fetch coastal land geometry, derive wave-coupling geometry when enabled, and plot the region context.

### 3. Required Static Data

Collect terrain, landcover, coastline, wave geometry, and SSURGO infiltration inputs for SFINCS. Marshfield has no Wflow static collection in this notebook.

### 4. Collected Input Plot

Plot the collected coastal static inputs and infiltration summaries.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

location_root = Path.cwd().parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
src_path = str(src_root)
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
importlib.invalidate_caches()
for module_name in [name for name in sys.modules if name == "sfincs_runs" or name.startswith("sfincs_runs.")]:
    del sys.modules[module_name]

import sfincs_runs.build_base.region_notebook as region

pd.set_option("display.max_colwidth", 140)


## 1. Parameters & Configuration


In [ ]:
runtime = region.load_coastal_region_setup_runtime(
    location_root,
    static_input_settings={"fetch_dem": True},
)

pd.Series(
    {
        "location": runtime.config["project"]["place_name"],
        "configured_bbox": runtime.config["static_sources"]["bbox"]["output"],
        "bbox_record_syntax": 'location_root / "data" / "static" / "aoi" / "bbox.geojson"',
        "static_record_syntax": 'location_root / "data" / "static" / "..."',
        "wflow_static_collection": "not applicable for coastal Marshfield",
    },
    name="region_setup_parameters",
)


## 2. Configured Coastal Region


In [ ]:
domain = region.build_configured_coastal_region_domain(runtime)

display(domain.summary)
region.plot_configured_coastal_region(runtime, domain)
plt.show()


## 3. Required Coastal Static Data


In [ ]:
static_data = region.collect_required_coastal_static_data(runtime, domain)

if static_data.dem_refresh is not None:
    display(static_data.dem_refresh)
display(static_data.terrain_landcover.summary)
display(static_data.coastal_region)
if static_data.wave_geometry is not None:
    display(static_data.wave_geometry)
display(static_data.ssurgo)
static_data.summary


## 4. Collected Input Plot


In [ ]:
coastal_static_qa = region.plot_collected_coastal_static_data(runtime, static_data, domain)

display(coastal_static_qa.hsg_summary)
coastal_static_qa.infiltration_summary
